In [16]:
import pandas as pd

customers = pd.read_csv("Clean Data/customers_cleaned.csv")
inventory = pd.read_csv("Clean Data/inventory_cleaned.csv")
order_items = pd.read_csv("Clean Data/order_items_cleaned.csv")
orders = pd.read_csv("Clean Data/orders_cleaned.csv")
payments = pd.read_csv("Clean Data/payments_cleaned.csv")
products = pd.read_csv("Clean Data/products_cleaned.csv")
returns = pd.read_csv("Clean Data/returns_cleaned.csv")
suppliers = pd.read_csv("Clean Data/suppliers_cleaned.csv")

In [17]:
# KPI 1: Gross Merchandise Value (GMV)

gmv = orders['final_amount'].sum()

print(f"Gross Merchandise Value (GMV): ₹{gmv:,.2f}")

Gross Merchandise Value (GMV): ₹3,147,385,544.58


In [18]:
### Observation

- Gross Merchandise Value (GMV) is ₹3,147,385,544.58.
- It represents the total value of all orders, including delivered, cancelled, returned, shipped, and processing orders.
- This KPI reflects the overall business transaction volume.

SyntaxError: invalid character '₹' (U+20B9) (2256686788.py, line 3)

In [ ]:
# KPI 2: Net Revenue

net_revenue = orders[orders['status'] == 'Delivered']['final_amount'].sum()

print (f"Net Revenue: ₹{net_revenue:,.2f}")

In [ ]:
### Observation

- Net Revenue is ₹2,053,858,611.96.
- It includes revenue only from delivered orders.
- This KPI represents the actual revenue earned after successful deliveries.

In [ ]:
# KPI 3: Average Order Value (AOV)

delivered_orders = orders[orders['status'] == 'Delivered'].shape[0]

aov = net_revenue / delivered_orders

print(f"Average Order Value (AOV): ₹{aov:,.2f}")

In [ ]:
### Observation

- Average Order Value (AOV) is ₹63,197.59.
- On average, each delivered order generated around ₹63.2K in revenue.
- A higher AOV indicates customers are spending more per successful order.

In [ ]:
# KPI 4: Cancellation Rate

cancelled_orders = orders[orders['status'] == 'Cancelled'].shape[0]

total_orders = orders.shape[0]

cancellation_rate = (cancelled_orders/total_orders) * 100

print(f"cancellation Rate: {cancellation_rate:.2f}%")

In [ ]:
### Observation

- Cancellation Rate is 11.99%.
- Nearly 12 out of every 100 orders were cancelled.
- This is higher than the target benchmark of 10%, indicating potential revenue loss.

In [ ]:
# KPI 5: Return Rate

returned_orders = len(returns)

delivered_orders = orders[orders['status'] == 'Delivered'].shape[0]

return_rate = (returned_orders / delivered_orders) * 100 

print (f"return Rate: {return_rate:.2f}%")

In [ ]:
### Observation

- Return Rate is 30.77%.
- Nearly 31 out of every 100 delivered orders were returned.
- This is much higher than the target benchmark of 8%, indicating significant product or customer satisfaction issues.

In [ ]:
customer_spend = orders.groupby('customer_id')['final_amount'].sum().reset_index()

customer_spend.head()

In [ ]:
clv_data = customer_spend.merge(
    customers[['customer_id', 'segment']],
    on='customer_id',
    how='left'
)

clv_data.head()

In [ ]:
# KPI 6: Customer Lifetime Value (CLV)

clv = customers.groupby('segment')['total_spent'].mean()

print("Customer Lifetime Value (CLV) by Segment:")
print(clv.round(2))

In [ ]:
### Observation

- Premium customers have the highest Customer Lifetime Value (CLV) at ₹532,735.00.
- Inactive customers have the lowest CLV at ₹32,249.88.
- Premium customers show substantially higher average lifetime spend compared with other customer segments.

In [ ]:
orders['order_date'] = pd.to_datetime(orders['order_date'])

In [ ]:
monthly_gmv = orders.groupby(
    orders['order_date'].dt.to_period('M')
)['final_amount'].sum()

monthly_gmv

In [ ]:
mom_growth = monthly_gmv.pct_change() * 100

print(mom_growth)

In [ ]:
# KPI 7: Month-over-Month Growth

mom_df = pd.DataFrame({
    'GMV': monthly_gmv,
    'MoM Growth (%)': mom_growth
})

mom_df

In [ ]:
### Observation

- Month-over-Month (MoM) Growth shows how business revenue changed compared to the previous month.
- Positive values indicate business growth, while negative values indicate a decline.
- The highest growth was observed in March 2025 (19.61%).
- The largest decline was observed in June 2025 (-23.74%).
- Overall, the business experienced both growth and decline across different months, indicating seasonal fluctuations.

In [ ]:
# KPI 8: Top Category Revenue Share

category_revenue = order_items.merge(
    products[['product_id', 'category']],
    on='product_id',
    how='left'
)

category_share = category_revenue.groupby('category_y')['total_price'].sum()

category_share = (category_share / category_share.sum()) * 100

print(category_share.sort_values(ascending=False))

In [ ]:
### Observation

- Electronics contributes the highest revenue share (57.41%).
- Sports & Fitness is the second-highest revenue-generating category (17.38%).
- Home & Kitchen contributes 8.54% of total revenue.
- The business is heavily dependent on Electronics for overall revenue.

In [ ]:
# KPI 9: Payment Failure Rate

failed_payments = payments[payments['status'] == 'Failed'].shape[0]

total_payments = payments.shape[0]

payment_failure_rate = (failed_payments / total_payments) * 100

print(f"Payment Failure Rate: {payment_failure_rate:.2f}%")

In [ ]:
### Observation

- Payment Failure Rate is 3.50%.
- Around 3.5 out of every 100 payment transactions failed.
- This indicates that the payment system is performing well with a low transaction failure rate.
- Maintaining this low failure rate helps improve customer experience and successful order completion.

In [ ]:
# KPI 10: Inventory Fill Rate

in_stock_skus = inventory[inventory['quantity_available'] > 0].shape[0]

total_skus = inventory.shape[0]

inventory_fill_rate = (in_stock_skus / total_skus) * 100

print(f"Inventory Fill Rate: {inventory_fill_rate:.2f}%")

In [ ]:
### Observation

- Inventory Fill Rate is 99.80%.
- This exceeds the target benchmark of 95%.
- Almost all products are available in stock, minimizing stock-out risk.
- This indicates excellent inventory management and ensures better customer satisfaction.

In [ ]:
# 1. Cohort Data Preparation

customers['registration_date'] = pd.to_datetime(
    customers['registration_date'],
    format='%d-%m-%Y',
    errors='coerce'
)

orders['order_date'] = pd.to_datetime(
    orders['order_date'],
    format='%d-%m-%Y',
    errors='coerce'
)

cohort_data = pd.merge(
    orders[['customer_id', 'order_date']],
    customers[['customer_id', 'registration_date']],
    on='customer_id',
    how='inner'
)

cohort_data = cohort_data.dropna(
    subset=['registration_date', 'order_date']
)

cohort_data['cohort_month'] = (
    cohort_data['registration_date'].dt.to_period('M')
)

cohort_data['order_month'] = (
    cohort_data['order_date'].dt.to_period('M')
)

cohort_data['cohort_index'] = (
    (cohort_data['order_month'].dt.year -
     cohort_data['cohort_month'].dt.year) * 12
    +
    (cohort_data['order_month'].dt.month -
     cohort_data['cohort_month'].dt.month)
)

cohort_data = cohort_data[
    cohort_data['cohort_index'] >= 0
]

cohort_table = cohort_data.groupby(
    ['cohort_month', 'cohort_index']
)['customer_id'].nunique().unstack(fill_value=0)

cohort_table

In [ ]:
# 2. Cohort Retention Calculation

# Total customers registered in each cohort
cohort_size = customers.groupby(
    customers['registration_date'].dt.to_period('M')
)['customer_id'].nunique()

# Calculate active customers for each cohort and month
cohort_retention = cohort_table.div(
    cohort_size,
    axis=0
) * 100

# Display retention percentage
cohort_retention.round(2)

In [ ]:
# 3. Customer Cohort Retention Heatmap

# Show first 12 months of retention
heatmap_data = cohort_retention.iloc[:, :13]

plt.figure(figsize=(14, 12))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    linewidths=0.5,
    mask=heatmap_data == 0
)

plt.title(
    "Customer Cohort Retention Analysis",
    fontsize=16
)

plt.xlabel(
    "Months Since Registration",
    fontsize=12
)

plt.ylabel(
    "Registration Cohort",
    fontsize=12
)

plt.tight_layout()

plt.savefig(
    "11_Customer_Cohort_Retention.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
### Cohort Analysis – Key Observation

Customer retention varies across registration cohorts and generally declines as the months since registration increase. 
Earlier cohorts show a wider retention pattern over time, while recent cohorts have fewer observable months due to the available data period. 
The cohort heatmap helps identify customer retention trends across different registration months.